In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive
Mounted at /content/drive


In [3]:
import os

NIDS_PATH = "/content/drive/MyDrive/NIDS/NIDS_processed_data"

print(os.listdir(NIDS_PATH))

['X_train_final.npy', 'X_test_final.npy', 'scaler.pkl', 'imputer.pkl', 'y_test_encoded.npy', 'protocol_encoder.pkl', 'y_train_encoded.npy', 'label_encoder.pkl']


In [6]:
import numpy as np
import pickle

X_train = np.load(f"{NIDS_PATH}/X_train_final.npy")
y_train = np.load(f"{NIDS_PATH}/y_train_encoded.npy")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nUnique encoded labels:")
print(np.unique(y_train))

print("\nNumber of unique classes:", len(np.unique(y_train)))

X_train shape: (1785444, 55)
y_train shape: (1785444,)

Unique encoded labels:
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]

Number of unique classes: 15


In [7]:
import joblib

label_encoder = joblib.load(
    f"{NIDS_PATH}/label_encoder.pkl"
)

print("Classes:")
for i, label in enumerate(label_encoder.classes_):
    print(i, "→", label)

Classes:
0 → Benign
1 → Bot
2 → DDoS
3 → DoS GoldenEye
4 → DoS Hulk
5 → DoS Slowhttptest
6 → DoS slowloris
7 → FTP-Patator
8 → Heartbleed
9 → Infiltration
10 → PortScan
11 → SSH-Patator
12 → Web Attack � Brute Force
13 → Web Attack � Sql Injection
14 → Web Attack � XSS


In [8]:
# Convert original 15-class labels into binary labels
# 0 = Normal (Benign)
# 1 = Attack (all other classes)
y_train_binary = np.where(y_train == 0, 0, 1)

unique, counts = np.unique(y_train_binary, return_counts=True)

print("Binary class distribution:")
for label, count in zip(unique, counts):
    name = "Normal" if label == 0 else "Attack"
    print(f"{label} ({name}): {count:,}")

Binary class distribution:
0 (Normal): 1,516,250
1 (Attack): 269,194


In [9]:
import numpy as np

RANDOM_SEED = 42

normal_mask = y_train_binary == 0
attack_mask = y_train_binary == 1

X_normal = X_train[normal_mask]
X_attack = X_train[attack_mask]

y_normal = y_train_binary[normal_mask]
y_attack = y_train_binary[attack_mask]

print("Before balancing:")
print("Normal:", len(y_normal))
print("Attack:", len(y_attack))

n_attack = len(y_attack)

# We want Normal : Attack = 2 : 1
n_normal_keep = 2 * n_attack

rng = np.random.default_rng(RANDOM_SEED)

normal_indices = rng.choice(
    len(X_normal),
    size=n_normal_keep,
    replace=False
)

X_normal_balanced = X_normal[normal_indices]
y_normal_balanced = y_normal[normal_indices]

X_train_balanced = np.concatenate(
    [X_normal_balanced, X_attack],
    axis=0
)

y_train_balanced = np.concatenate(
    [y_normal_balanced, y_attack],
    axis=0
)

shuffle_indices = rng.permutation(len(y_train_balanced))

X_train_balanced = X_train_balanced[shuffle_indices]
y_train_balanced = y_train_balanced[shuffle_indices]

unique, counts = np.unique(
    y_train_balanced,
    return_counts=True
)

print("\nAfter balancing:")

for label, count in zip(unique, counts):
    name = "Normal" if label == 0 else "Attack"
    print(f"{label} ({name}): {count:,}")

print("\nFinal X_train shape:", X_train_balanced.shape)
print("Final y_train shape:", y_train_balanced.shape)

Before balancing:
Normal: 1516250
Attack: 269194

After balancing:
0 (Normal): 538,388
1 (Attack): 269,194

Final X_train shape: (807582, 55)
Final y_train shape: (807582,)


In [10]:
normal_count = np.sum(y_train_balanced == 0)
attack_count = np.sum(y_train_balanced == 1)

print("Class distribution:")
print(f"Normal : {normal_count:,}")
print(f"Attack : {attack_count:,}")

print("\nNormal:Attack ratio:",
      normal_count / attack_count)

print("\nShapes:")
print("X_train_balanced:", X_train_balanced.shape)
print("y_train_balanced:", y_train_balanced.shape)

print("\nNaN check:")
print("NaN values:", np.isnan(X_train_balanced).sum())

print("\nInfinite value check:")
print("Infinite values:", np.isinf(X_train_balanced).sum())

print("\nData type:")
print(X_train_balanced.dtype)

Class distribution:
Normal : 538,388
Attack : 269,194

Normal:Attack ratio: 2.0

Shapes:
X_train_balanced: (807582, 55)
y_train_balanced: (807582,)

NaN check:
NaN values: 0

Infinite value check:
Infinite values: 0

Data type:
float64


In [11]:
np.save(
    f"{NIDS_PATH}/X_train_stage1_balanced.npy",
    X_train_balanced
)

np.save(
    f"{NIDS_PATH}/y_train_stage1_binary.npy",
    y_train_balanced
)

print("Stage 1 balanced training data saved successfully!")

print("\nSaved files:")
print("X_train_stage1_balanced.npy")
print("y_train_stage1_binary.npy")

Stage 1 balanced training data saved successfully!

Saved files:
X_train_stage1_balanced.npy
y_train_stage1_binary.npy
